In [18]:
from crewai import Agent, Task, Crew, Process, LLM
from langchain_ollama import OllamaLLM
from pydantic import BaseModel

In [19]:
class task(BaseModel):
    taskID: int
    task_title: str
    task_description: str
    difficulty_level: str
    rewards: int

In [20]:
llm = LLM(
    model="ollama/openhermes",
    base_url="http://localhost:11434"
    )

manager_llm = LLM(
    model = "ollama/llama3.2",
    base_url="http://localhost:11434"
)

In [21]:
taskGiver = Agent (
    role = "Task Giver",
    goal = (
        "Generate creative, age-appropriate tasks for children aged 6 to 12 that can be done at home. "
        "Tasks should be realistic and safe for children to accomplish in the real world, specifically in Singapore. "
        "Include the following details for each task: "
        "- Task title (brief but descriptive). "
        "- Task description (clear and detailed instructions). "
        "- Difficulty level (easy, medium, or hard). "
        "- Rewards (based on difficulty, ranging from $1 to $10). "
        "Ensure the tasks encourage learning, responsibility, or creativity while being achievable by children."
        ),
    backstory=(
        "I am a caring mother who wants to give my child meaningful and fun tasks to help out around the house. "
        "I want the tasks to teach valuable skills while keeping my child engaged."
        ),
    llm = llm,
    verbose=True,
    allow_delegation=False
)

taskValidator = Agent (
    role = "Task Validator",
    goal = (
        "Review and validate the tasks provided by the Task Giver. "
        "Ensure each task is age-appropriate, safe, and feasible for children aged 6 to 12 to accomplish at home. "
        "Ensure the task is suitable for children in Singapore. "
        "Summarize the tasks in the specified JSON format, ensuring that each task includes the following fields: "
        "- 'taskID' (unique identifier). "
        "- 'task_title' (brief title). "
        "- 'task_description' (detailed instructions). "
        "- 'difficulty_level' (easy, medium, hard). "
        "- 'rewards' (amount between $1 and $10, based on difficulty)."
        ),
    backstory=(
        "I am a thoughtful father who reviews tasks to ensure they are suitable for my child. "
        "I aim to validate and summarize the tasks in a structured format."
        ),
    llm = llm,
    verbose=True,
    allow_delegation=False
)

task1 = Task(
    description=(
        "Generate age-appropriate, safe, and realistic tasks for children aged 6 to 12 to do at home in Singapore. "
        "The output should include a detailed task description, difficulty level (easy, medium, or hard), and reward amount "
        "($1 to $10 based on task difficulty)."
        ),
    agent=taskGiver,
    expected_output=(
        "A JSON object with the fields: 'task_title', 'task_description', 'difficulty_level', and 'rewards'. "
        "Example: {'task_title': 'Tidy Your Room', 'task_description': 'Organize toys, books, and clothes in your room.', "
        "'difficulty_level': 'easy', 'rewards': 2}"
        )
)

task2 = Task(
    description=(
        #"Validate the tasks given to the child, and make sure it is doable by a 6-12 year old child"
        "Validate the tasks provided by the Task Giver. Ensure they are appropriate for children in Singapore aged 6 to 12, "
        "safe, and feasible. Summarize the task in the following JSON format: "
        "{'taskID': int, 'task_title': str, 'task_description': str, 'difficulty_level': str, 'rewards': int}."
        ),
    agent=taskValidator,
    expected_output=(
        #"A JSON object with 'taskID', 'task_title' , 'task_description', 'difficulty_level' and 'rewards' fields."
        "A JSON object with validated task details, including fields: 'taskID', 'task_title', 'task_description', "
        "'difficulty_level', and 'rewards'."
        ),
    output_json=task
)


#manager = Agent(
#    role="Task Manager",
#    goal=(
#        "Coordinate and oversee the task creation and validation process between the Task Giver and Task Validator agents. "
#        "Ensure that the tasks generated by the Task Giver are passed to the Task Validator for review. "
#        "Keep track of the progress and ensure all tasks meet the required standards for age-appropriateness, safety, and feasibility. "
#        "Facilitate smooth communication between agents, resolve any conflicts or discrepancies, and ensure that the finalized tasks are well-organized and ready for presentation."
#    ),
#    backstory=(
#        "I am an organized and efficient manager responsible for ensuring that all tasks are generated, validated, and finalized seamlessly. "
#        "I work closely with both the Task Giver and Task Validator to ensure that tasks meet the required quality standards and are suitable for children aged 6 to 12. "
#        "My role is to ensure that the entire process runs smoothly and efficiently."
#    ),
#    llm=manager_llm,
#    verbose=True,
#    allow_delegation=True
#)


In [22]:
crew = Crew (
    agents=[taskGiver, taskValidator],
    tasks=[task1, task2],
    verbose=True,
    #manager_agent=manager,
    process=Process.sequential,
)

Traceback (most recent call last):
  File "c:\Users\jjie\.vscode\extensions\ms-python.python-2024.22.2-win32-x64\python_files\python_server.py", line 133, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\jjie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\main.py", line 214, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for Crew
  Value error, Please provide an OpenAI API key. You can get one at https://platform.openai.com/account/api-keys [type=value_error, input_value={'agents': [Agent(role=Ta...ntial'>, 'memory': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error



In [23]:
result = crew.kickoff()
# check the type of the result
print(type(result))
print(result)
tasks = result['task_title'] 
print(tasks)

Traceback (most recent call last):
  File "c:\Users\jjie\.vscode\extensions\ms-python.python-2024.22.2-win32-x64\python_files\python_server.py", line 133, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'crew' is not defined. Did you mean: 'Crew'?

